In [25]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file.lower() == "tegenploeg.html":
            print(os.path.join(root, file))

./tegenploeg.html


In [26]:
TEGENPLOEG = './tegenploeg.html'

with open(TEGENPLOEG, "r", encoding="utf-8") as f:
    html_content = f.read()

html_content[:500]  # Shows the first 500 characters


'\n      <div class="tableContainer gameStatsContainer">\n    \n    <div class="bootstrap-table bootstrap5">\n      <div class="fixed-table-toolbar"><div class="bs-bars float-left"><div id="table-toolbar-Home">\n        <a href="#" aria-current="page" data-bs-toggle="modal" data-bs-target="#tableInfoGame" style="vertical-align: sub;"><i class="bi bi-info-circle-fill"></i> Meer informatie</a>\n    </div></div><div class="columns columns-right btn-group float-right"><div class="keep-open btn-group">\n    '

# functions

In [27]:
def fetch_player_ALL(SPELERID):
    url = "https://app.basketballstatsvlaanderen.be/players/" + SPELERID + "season=2425"
    url = 'https://app.basketballstatsvlaanderen.be/players/' + SPELERID
    import requests
    import re
    from bs4 import BeautifulSoup
    import json
    import pandas as pd

    response = requests.get(url)
    html_doc = response.text

    soup = BeautifulSoup(html_doc, 'html.parser')
    games_data = []

    for script in soup.find_all('script'):
        if script.string and "var games = [" in script.string:
            match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
            if match:
                games_str = match.group(0).replace('var games =', '').strip(' ;')
                try:
                    games_data = json.loads(games_str)
                    break
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    # Extract subfields from 'GameTeam' if present
    for game in games_data:
        if 'GameTeam' in game and isinstance(game['GameTeam'], dict):
            game['GameTeam_TEAM'] = game['GameTeam'].get('Name', {})
            game['GameTeam_GameResult'] = game['GameTeam'].get('Game', {}).get('Result')
            game['GameTeam_GameGuid'] = game['GameTeam'].get('Game', {}).get('Guid')
            game['GameTeam_GameDate'] = game['GameTeam'].get('Game', {}).get('Date')
            game['GameTeam_AwayTeamName'] = game['GameTeam'].get('Game', {}).get('AwayTeam', {}).get('Name')
            game['GameTeam_HomeTeamName'] = game['GameTeam'].get('Game', {}).get('HomeTeam', {}).get('Name')

            # Add AGE column by trimming after the 2nd last space in GameTeam_TEAM
            for game in games_data:
                team_name = game.get('GameTeam_TEAM', '')
                if isinstance(team_name, str):
                    parts = team_name.split(' ')
                    if len(parts) > 2:
                        game['AGE'] = ' '.join(parts[-2:])
                    else:
                        game['AGE'] = team_name
                else:
                    game['AGE'] = None
    

    # filtered_games_data = [
    #     {k: game[k] for k in fields_to_keep if k in game}
    #     for game in games_data
    # ]

    df_games = pd.DataFrame(games_data)
    
    # df_games = df_games.drop(columns=['id', 'GameTeamId', 'Stints', 'createdAt', 'updatedAt', 'GameTeam'])
    return df_games


In [28]:
fetch_player_ALL('BVBL748910')


ModuleNotFoundError: No module named 'pandas'

In [ ]:
def FETCH_PLAYERS_AVG(SPELERID):
    df = fetch_player_ALL(SPELERID)
    # Unnest AGE from GameTeam column if not already present
    if 'AGE' not in df.columns:
        df['AGE'] = df['GameTeam'].apply(lambda x: ' '.join(x['Name'].split(' ')[-2:]) if isinstance(x, dict) and 'Name' in x else None)

    grouped = df.groupby(['Name', 'AGE'])
    result = grouped.agg({
        'TotalMinutes': 'mean',
        'NormalizedMinutes': 'mean',
        'FreeThrows': 'mean',
        'FieldGoals': 'mean',
        'ThreePointers': 'mean',
        'TotalScore': ['min', 'mean', 'median', 'max' , 'count'],
        'PlusMinus': 'mean',
        'Faults': 'mean',
        'Plus': 'mean',
        'Minus': 'mean',
       
    })

    # Flatten MultiIndex columns
    result.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in result.columns.values]
    # Rename and rearrange columns to match the desired order and names
    result = result.rename(columns={
         'TotalScore_count': 'WEDSTRIJDEN',
        
        'TotalMinutes_mean': 'Avg TotalMinutes',
        'NormalizedMinutes_mean': 'Avg NormalizedMinutes',
        'FreeThrows_mean': 'Avg FreeThrows',
        'FieldGoals_mean': 'Avg FieldGoals',
        'ThreePointers_mean': 'Avg ThreePointers',
        'TotalScore_min': 'Min TotalScore',
        'TotalScore_mean': 'Avg TotalScore',
        'TotalScore_median': 'Median TotalScore',
        'TotalScore_max': 'Max TotalScore',
       'Faults_mean': 'Avg Faults',
        'PlusMinus_mean': 'Avg PlusMinus',
        
        'Plus_mean': 'Avg Plus',
        'Minus_mean': 'Avg Minus'
    })

    # Reorder columns to match the desired template
    desired_order = [
        'WEDSTRIJDEN',
        'Avg TotalMinutes',
        'Avg NormalizedMinutes',
        'Avg FreeThrows',
        'Avg FieldGoals',
        'Avg ThreePointers',
        'Min TotalScore',
        'Avg TotalScore',
        'Median TotalScore',
        'Max TotalScore',
        
        'Avg PlusMinus',
        'Avg Faults',
        'Avg Plus',
        'Avg Minus'
    ]
    # Keep index columns (Name, AGE) at the front
    result = result.reset_index()[['Name', 'AGE'] + desired_order]
    # Round all columns containing 'Avg' to 1 decimal place
    avg_cols = [col for col in result.columns if 'Avg' in col]
    result[avg_cols] = result[avg_cols].round(1)
    return result




In [32]:
import pandas as pd

In [33]:
import pandas as pd
from bs4 import BeautifulSoup

# Read the HTML file and parse with BeautifulSoup
with open(TEGENPLOEG, 'r', encoding='utf-8') as f:
    html_doc = f.read()
soup = BeautifulSoup(html_doc, 'html.parser')

# Extract player data directly from soup (tbody)
tbody = soup.find('tbody')
rows = tbody.find_all('tr') if tbody else []

player_list = []
for row in rows:
    cols = row.find_all('td')
    if len(cols) >= 2:
        # Extract jersey number (remove icon if present)
        jersey_text = cols[0].get_text(strip=True)
        jersey_number = pd.to_numeric(jersey_text.split()[-1], errors='coerce')
        name = cols[1].get_text(strip=True)
        if pd.notna(jersey_number) and name:
            player_list.append({'jersey_number': int(jersey_number), 'name': name})

player_data = pd.DataFrame(player_list)
player_data

,jersey_number,name
0,1,Wout Van den Bossche
1,3,Corneel Vanceulebroeck
2,4,Rogelio Love
3,7,Briek Van Gaever
4,9,Arne De Winne
5,10,Wannes François
6,12,Remi Van Hulle
7,13,Jãnis Ramma
8,15,Kasper De Ridder
9,20,Tibo Van Hecke


In [35]:
import json
import pandas as pd

# Load the ALLPLAYERS.json file
with open('./ALLPLAYERS.json', 'r', encoding='utf-8') as f:
    allplayers_data = json.load(f)

# Convert to DataFrame and expand 'rows' if necessary
allplayers_df = pd.DataFrame(allplayers_data)
if isinstance(allplayers_df['rows'].iloc[0], dict):
    rows_expanded = allplayers_df['rows'].apply(pd.Series)
    allplayers_df = pd.concat([allplayers_df.drop(columns=['rows']), rows_expanded], axis=1)

# Example: find Guid, LidNr, Name for a given player name
def find_player_info(player_name):
    match = allplayers_df[allplayers_df['Name'] == player_name]
    if not match.empty:
        return match[['Guid', 'LidNr', 'Name']].iloc[0].to_dict()
    else:
        return None


player_data['Guid'] = player_data['name'].map(lambda n: find_player_info(n)['Guid'] if find_player_info(n) else None)
player_data['LidNr'] = player_data['name'].map(lambda n: find_player_info(n)['LidNr'] if find_player_info(n) else None)
# print(player_data)

In [41]:
player_data
# Collect all stats for each player using their Guid
all_stats = []

for guid in player_data['Guid']:
    stats = fetch_player_ALL(guid)
    stats['Guid'] = guid  # Add Guid column if not present
    all_stats.append(stats)

all_stats_df = pd.concat(all_stats, ignore_index=True)

In [43]:
all_stats_df.to_excel('gent_allstats.xlsx', index=False)

In [ ]:
# Add columns from FETCH_PLAYERS_AVG function to player_data DataFrame
for index, row in player_data.iterrows():
    guid = row['Guid']
    if pd.notna(guid):
        try:
            # Get player stats using the FETCH_PLAYERS_AVG function
            player_stats = FETCH_PLAYERS_AVG(guid)
            
            if not player_stats.empty:
                # Get the first row of stats (assuming one player per Guid)
                stats_row = player_stats.iloc[0]
                
                # Add all stats columns to player_data
                for col in player_stats.columns:
                    if col not in ['Name', 'AGE']:  # Skip duplicate columns
                        player_data.loc[index, col] = stats_row[col]
        except Exception as e:
            print(f"Error fetching stats for {row['name']} (Guid: {guid}): {e}")

print(f"Updated player_data with stats for {len(player_data)} players")
player_data

Updated player_data with stats for 10 players


,jersey_number,name,Guid,LidNr,WEDSTRIJDEN,Avg TotalMinutes,Avg NormalizedMinutes,Avg FreeThrows,Avg FieldGoals,Avg ThreePointers,Min TotalScore,Avg TotalScore,Median TotalScore,Max TotalScore,Avg PlusMinus,Avg Faults,Avg Plus,Avg Minus
0,4,Lucas Verbeke,BVBL740842,740842,7.0,18.1,19.0,0.1,2.3,0.0,2.0,2.4,2.0,5.0,-11.3,0.6,18.1,-29.4
1,5,Seydina-Madione-Laye Diagne,BVBL737985,737985,9.0,11.7,13.6,0.7,1.8,0.0,0.0,2.4,1.0,9.0,0.6,0.8,25.6,-25.0
2,7,Robbe Verhelle,BVBL706242,706242,10.0,22.6,26.1,0.3,4.8,3.9,3.0,9.0,7.0,19.0,3.3,2.4,54.4,-51.1
3,8,Ewout Devos,BVBL706235,706235,10.0,18.8,21.8,1.4,2.8,2.7,2.0,6.9,6.0,16.0,3.2,2.3,44.7,-41.5
4,9,Jonas Van Camp,BVBL713112,713112,10.0,15.8,18.3,0.9,8.0,2.1,6.0,11.0,9.5,20.0,4.4,3.5,38.0,-33.6
5,12,Jakob De Lameillieure,BVBL668549,668556,9.0,24.0,28.0,3.7,12.4,3.7,5.0,19.8,18.0,40.0,13.7,2.6,60.4,-46.8
6,14,Viktor Verlinde,BVBL669431,669438,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.0,1.0,0.0,-4.0
7,15,Thomas Defeau,BVBL703499,703499,10.0,17.4,20.2,1.6,4.8,0.3,2.0,6.7,4.0,15.0,-5.5,2.7,38.9,-44.4
8,16,Lowie Scheirlynck,BVBL664524,664531,10.0,17.2,19.8,1.9,8.8,0.0,4.0,10.7,8.5,24.0,5.4,2.2,41.0,-35.6
9,20,Xander Geldhof,BVBL666427,666434,9.0,14.0,16.1,0.8,4.4,0.0,1.0,5.2,5.0,12.0,-5.4,1.0,28.3,-33.8


In [ ]:
player_data.to_excel('player_stats.xlsx', index=False)

## FUNCTIONS
Let's extract player data from the identified tbody elements: